In [1]:
import warnings
warnings.filterwarnings('ignore')
!pip install -r requirement.txt

In [2]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Retrieve the secret key stored in Colab Secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 2. Set the environment variable (LangChain automatically looks for GOOGLE_API_KEY)
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# 3. Initialize Gemini via LangChain
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

# 4. Run inference using LangChain's invoke method
response = llm.invoke("Hello! Confirm connection.")
print(response.content)

[{'type': 'text', 'text': 'Connection confirmed! Hello! How can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0T72791BcmcnJvZQFkvQiC/tJjSGd4SINH+NeU9Hvq1+E+ASnSEJNY/v78sUPeEXhicK3zXyM2iIlVGCofjjIYoaDEbb6ILisXsKB3J7A6kiczkDONvaloVqo='}}]


In [3]:
import bs4
import langchain
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Embeddings & Vector Store
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 3. Model & Prompts
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

In [4]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

url = "https://lilianweng.github.io/posts/2023-06-23-agent/"

loader = WebBaseLoader(
    web_paths=(url,),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 1. Split document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# 2. Embed and store in Chroma
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

In [32]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [33]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [34]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)

In [35]:
from langchain_classic.chains import create_retrieval_chain

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [36]:
response = rag_chain.invoke({"input": "What is task decomposition?"})

print(response["answer"])

Task decomposition is the process of breaking down a complicated task into smaller, manageable, and simpler steps. It can be performed using simple LLM prompting, task-specific instructions, or human inputs. This approach helps an agent plan ahead and enhances performance on complex tasks.


In [37]:
rag_chain.invoke({"input":"How do we achieve it"})

{'input': 'How do we achieve it',
 'context': [Document(id='8457cf32-6504-4515-a657-c5017e4190d3', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content="(3) Task execution: Expert models execute on the specific tasks and log results.\nInstruction:\n\nWith the input and the inference results, the AI assistant needs to describe the process and results. The previous stages can be formed as - User Input: {{ User Input }}, Task Planning: {{ Tasks }}, Model Selection: {{ Model Assignment }}, Task Execution: {{ Predictions }}. You must first answer the user's request in a straightforward manner. Then describe the task process and show your analysis and model inference results to the user in the first person. If inference results contain a file path, must tell the user the complete file path."),
  Document(id='be75b5f7-89ca-4904-b38f-bb7fc9663bf2', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content="(3) Task execution:

In [38]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

In [39]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

In [40]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [41]:
from langchain_classic.chains import create_history_aware_retriever

history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt
)

In [42]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [43]:
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

In [44]:
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [45]:
from langchain_core.messages import AIMessage, HumanMessage

In [46]:
chat_history = []

# 3. Define Question 1
question_1 = "What is task decomposition?"

# 4. Invoke rag_chain with input and chat_history
response = rag_chain.invoke({
    "input": question_1,
    "chat_history": chat_history
})

print("Answer:\n", response["answer"])

# 5. Append exchange to chat_history for subsequent turns
chat_history.extend([
    HumanMessage(content=question_1),
    AIMessage(content=response["answer"])
])

Answer:
 Task decomposition is the process of breaking down a complicated, hard task into smaller, manageable, and simpler steps. This can be achieved through LLM prompting methods, task-specific instructions, or human inputs. Techniques like Chain of Thought and Tree of Thoughts help facilitate this process by encouraging models to think step by step or explore multiple reasoning possibilities.


In [48]:
embeddings = GoogleGenerativeAIEmbeddings(model="text-embedding-004")